# Step 1: Setup up dataset AND constant values [DONE for wiki, need to do for ambig]
# Step 2: Convert Query to n-Query [Maybe, need more testing issues - how to convert llm reposon to n-query?]
# Step 3: Create a function to combine n-query doc ranking to k-doc ranking [DONE]
# Step 4: Setup the rag pipeline

In [2]:
import pyterrier as pt

from code_base.wiki_loader_pyt import wiki_loader_pyt

# Constants
DATA_PATH = "/mnt/c/Users/prath/Documents/Uni(Tudelft)/MSc/MasterThesis/data/2wikimultihopQA"
SPLIT = "dev"

# Load dataframes
docs_df, topics_df, qrels_df = wiki_loader_pyt(DATA_PATH, SPLIT)

Processing corpus...


100%|██████████| 563424/563424 [00:00<00:00, 2033973.01it/s]


Created docs_df with 495034 documents.

Processing dev set for topics and qrels...


100%|██████████| 12576/12576 [00:00<00:00, 242546.54it/s]


Created topics_df with 12576 queries.
Created qrels_df with 125760 relevance judgments.


In [2]:
docs_df = docs_df.dropna(subset=['docno', 'text'])
docs_df['docno'] = docs_df['docno'].astype(str)
docs_df = docs_df[docs_df['text'].str.strip() != '']
docs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 495034 entries, 0 to 495033
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   docno   495034 non-null  object
 1   text    495034 non-null  object
dtypes: object(2)
memory usage: 7.6+ MB


In [3]:
#Uncomment if you want to reindex
# indexer = pt.IterDictIndexer("./index/wiki_musique_index", overwrite=True, meta={'docno': 256, 'text': 4096})
# index_ref = indexer.index(docs_df.to_dict(orient='records'))

index_path = "./index/wiki_musique_index/data.properties"  # path to the Terrier index metadata
index_ref = pt.IndexFactory.of(index_path)


print(index_ref)

Java started (triggered by IndexFactory.of) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


Index created at: <org.terrier.structures.Index at 0x7598199bcb30 jclass=org/terrier/structures/Index jself=<LocalRef obj=0x325534a8 at 0x7598130dca90>>


In [9]:
bm25 = pt.terrier.Retriever(index_ref, wmodel="BM25", controls={}) #TODO fix this lexical issues without change the query

experiment = pt.Experiment(
    [bm25],
    topics_df,
    qrels_df,
    eval_metrics=["map", "ndcg",],
    names=["BM25"]
)

JavaException: JVM exception occurred: org.terrier.querying.parser.QueryParserException: Failed to process qid 8813f87c0bdd11eba7f7acde48001122 'Who is the mother of the director of film Polish-Russian War (Film)?' -- Lexical error at line 1, column 69.  Encountered: <EOF> after : ""
org.terrier.querying.parser.QueryParserException: Failed to process qid 8813f87c0bdd11eba7f7acde48001122 'Who is the mother of the director of film Polish-Russian War (Film)?' -- Lexical error at line 1, column 69.  Encountered: <EOF> after : ""
	org.terrier.querying.parser.QueryParser.parseQuery(QueryParser.java:81)
	org.terrier.querying.parser.QueryParser.parseQuery(QueryParser.java:50)
	org.terrier.querying.TerrierQLParser.process(TerrierQLParser.java:39)
	org.terrier.querying.LocalManager.runSearchRequest(LocalManager.java:895)
	Caused by:
	org.terrier.querying.parser.TokenMgrError: Lexical error at line 1, column 69.  Encountered: <EOF> after : ""
	org.terrier.querying.parser.TerrierQLParserTokenManag

retrival dataset - Bright dataset

